In [ ]:
# -*- coding: utf-8 -*-
"""
Refactored TabNet 5-Level Configuration Experiment Script.

This script trains and evaluates five different configurations of a TabNet model
on brain voxel data. It features a robust, callback-based training loop for
detailed monitoring, early stopping, and model checkpointing.

Key Improvements in this Version:
- Centralized Control: A single `SupervisingCallback` class now manages all
  in-training events (evaluation, logging, early stopping, checkpointing),
  simplifying the training loop.
- Reduced Redundancy: Removed duplicate classes and methods. The logic for
  evaluation and reporting is now clearly separated and more efficient.
- Improved Structure: Classes have clearer responsibilities, making the code
  easier to understand, maintain, and extend.
- Simplified Main Execution: The main script block is significantly cleaner,
  with a clear workflow for running experiments and generating reports.
"""

import os
import json
import time
import warnings
from copy import deepcopy

import h5py
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, cohen_kappa_score,
                             balanced_accuracy_score)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from pytorch_tabnet.tab_model import TabNetClassifier
from pytorch_tabnet.metrics import Metric
# The 'Callback' import is moved down to be closer to its usage for robustness.

# --- Configuration ---

# Ignore warnings for cleaner output
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Set device and base export path
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EXPORT_PATH = './tabnet_refactored_experiment/'

# Create necessary directories
os.makedirs(os.path.join(EXPORT_PATH, 'visualizations'), exist_ok=True)
os.makedirs(os.path.join(EXPORT_PATH, 'models'), exist_ok=True)
os.makedirs(os.path.join(EXPORT_PATH, 'logs'), exist_ok=True)

print(f"🖥️ Using device: {DEVICE}")
print(f"✅ Environment setup complete. Files will be saved to '{EXPORT_PATH}'")


# --- Model and Training Configurations ---

TABNET_COMPLETE_CONFIGS = {
    'micro_config': {
        'n_d': 16, 'n_a': 16, 'n_steps': 2, 'gamma': 1.8, 'n_independent': 1, 'n_shared': 1,
        'lambda_sparse': 8e-3, 'mask_type': 'sparsemax', 'learning_rate': 8e-3, 'weight_decay': 5e-4,
        'momentum': 0.4, 'clip_value': 2.5, 'batch_size': 256, 'virtual_batch_size_ratio': 0.4,
        'max_epochs': 150, 'patience': 25, 'scheduler_patience': 8, 'scheduler_factor': 0.7,
        'estimated_params': '~1M'
    },
    'small_config': {
        'n_d': 32, 'n_a': 32, 'n_steps': 3, 'gamma': 1.6, 'n_independent': 2, 'n_shared': 2,
        'lambda_sparse': 3e-3, 'mask_type': 'sparsemax', 'learning_rate': 5e-3, 'weight_decay': 3e-4,
        'momentum': 0.35, 'clip_value': 2.0, 'batch_size': 512, 'virtual_batch_size_ratio': 0.35,
        'max_epochs': 180, 'patience': 30, 'scheduler_patience': 10, 'scheduler_factor': 0.6,
        'estimated_params': '~3M'
    },
    'medium_config': {
        'n_d': 64, 'n_a': 64, 'n_steps': 4, 'gamma': 1.4, 'n_independent': 3, 'n_shared': 2,
        'lambda_sparse': 8e-4, 'mask_type': 'sparsemax', 'learning_rate': 3e-3, 'weight_decay': 2e-4,
        'momentum': 0.3, 'clip_value': 1.8, 'batch_size': 512, 'virtual_batch_size_ratio': 0.3,
        'max_epochs': 200, 'patience': 35, 'scheduler_patience': 12, 'scheduler_factor': 0.5,
        'estimated_params': '~8M'
    },
    'large_config': {
        'n_d': 128, 'n_a': 128, 'n_steps': 5, 'gamma': 1.25, 'n_independent': 4, 'n_shared': 3,
        'lambda_sparse': 3e-4, 'mask_type': 'sparsemax', 'learning_rate': 2e-3, 'weight_decay': 1e-4,
        'momentum': 0.25, 'clip_value': 1.5, 'batch_size': 1024, 'virtual_batch_size_ratio': 0.25,
        'max_epochs': 150, 'patience': 40, 'scheduler_patience': 15, 'scheduler_factor': 0.4,
        'estimated_params': '~20M'
    },
    'xlarge_config': {
        'n_d': 256, 'n_a': 256, 'n_steps': 6, 'gamma': 1.15, 'n_independent': 5, 'n_shared': 3,
        'lambda_sparse': 1e-4, 'mask_type': 'sparsemax', 'learning_rate': 1.5e-3, 'weight_decay': 8e-5,
        'momentum': 0.2, 'clip_value': 1.2, 'batch_size': 1024, 'virtual_batch_size_ratio': 0.2,
        'max_epochs': 150, 'patience': 45, 'scheduler_patience': 18, 'scheduler_factor': 0.3,
        'estimated_params': '~35M (vs 4x4096 FC)'
    }
}
print("📋 TabNet configurations defined.")


# --- Utility Functions ---

def calculate_cross_entropy_loss(y_true, y_pred_proba):
    """Safely calculates cross-entropy loss."""
    # Clip probabilities to avoid log(0) error
    y_pred_proba = np.clip(y_pred_proba, 1e-15, 1 - 1e-15)
    # Select the probability of the true class for each sample
    correct_logprobs = np.log(y_pred_proba[np.arange(len(y_true)), y_true])
    return -np.mean(correct_logprobs)

# --- Data Loading and Preprocessing ---

def load_and_preprocess_data(file_path):
    """Loads and preprocesses the brain voxel data."""
    print("\n📂 Loading and preprocessing data...")
    with h5py.File(file_path, 'r') as f:
        train_data = np.array(f['data']).transpose()
        train_region = np.array(f['region']).transpose()
        prob_idx = np.array(f['prob_idx']).transpose()

    print(f"  - Original data shape: {train_data.shape}")
    print(f"  - Original labels shape: {train_region.shape}")

    # Split data based on subject index (38 is test subject)
    test_indices = np.where(prob_idx == 38)[0]
    train_val_indices = np.where(prob_idx != 38)[0]

    X_test_raw = train_data[test_indices, :]
    y_test_raw = train_region[test_indices, :]
    X_train_val_raw = train_data[train_val_indices, :]
    y_train_val_raw = train_region[train_val_indices, :]

    # Split training data into training and validation sets
    y_train_val_int = np.argmax(y_train_val_raw, axis=1)
    X_train_raw, X_val_raw, y_train_raw, y_val_raw = train_test_split(
        X_train_val_raw, y_train_val_raw,
        test_size=0.2, random_state=42, stratify=y_train_val_int
    )

    # Standardize features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_val_scaled = scaler.transform(X_val_raw)
    X_test_scaled = scaler.transform(X_test_raw)

    # Convert one-hot labels to integer format for TabNet
    y_train_int = np.argmax(y_train_raw, axis=1)
    y_val_int = np.argmax(y_val_raw, axis=1)
    y_test_int = np.argmax(y_test_raw, axis=1)

    print("  - Data splits:")
    print(f"    - Training set:   {X_train_scaled.shape}, Labels: {y_train_int.shape}")
    print(f"    - Validation set: {X_val_scaled.shape}, Labels: {y_val_int.shape}")
    print(f"    - Test set:       {X_test_scaled.shape}, Labels: {y_test_int.shape}")
    print(f"  - Feature count: {X_train_scaled.shape[1]}")
    print(f"  - Class count: {len(np.unique(y_train_int))}")
    print("✅ Data loading complete.")

    return X_train_scaled, y_train_int, X_val_scaled, y_val_int, X_test_scaled, y_test_int


# --- Core Classes for Training and Evaluation ---
from pytorch_tabnet.callbacks import Callback

class SupervisingCallback(Callback):
    """
    An all-in-one callback for monitoring, early stopping, and checkpointing.
    """
    def __init__(self, X_train, y_train, X_val, y_val, X_test, y_test,
                 config, export_path, checkpoint_frequency=5):
        self.X_train, self.y_train = X_train, y_train
        self.X_val, self.y_val = X_val, y_val
        self.X_test, self.y_test = X_test, y_test
        
        self.config = config
        self.config_name = self.config.get('config_name', 'default_config')
        self.export_path = export_path
        self.checkpoint_frequency = checkpoint_frequency
        self.checkpoint_dir = os.path.join(export_path, 'models', self.config_name)
        os.makedirs(self.checkpoint_dir, exist_ok=True)
        
        # Early Stopping parameters
        self.patience = self.config.get('patience', 25)
        self.min_delta = 1e-5
        self.wait = 0
        self.best_val_f1 = -np.inf
        self.stopped_epoch = 0
        
        self.history = {k: [] for k in [
            'epoch', 'train_loss', 'train_f1', 'val_loss', 'val_f1', 
            'test_loss', 'test_f1', 'train_acc', 'val_acc', 'test_acc', 'lr'
        ]}
        
        self.start_time = time.time()
        print(f"  - Callback initialized for '{self.config_name}'. Patience={self.patience}, Checkpoint Freq={self.checkpoint_frequency}.")

    def on_epoch_end(self, epoch, logs=None):
        """Actions to perform at the end of each epoch."""
        model = self.model  # Access model directly via self.model in TabNet callbacks

        # --- 1. Comprehensive Evaluation ---
        train_proba = model.predict_proba(self.X_train)
        val_proba = model.predict_proba(self.X_val)
        test_proba = model.predict_proba(self.X_test)
        
        train_preds = np.argmax(train_proba, axis=1)
        val_preds = np.argmax(val_proba, axis=1)
        test_preds = np.argmax(test_proba, axis=1)

        metrics = {
            'train_f1': f1_score(self.y_train, train_preds, average='macro'),
            'val_f1': f1_score(self.y_val, val_preds, average='macro'),
            'test_f1': f1_score(self.y_test, test_preds, average='macro'),
            'train_loss': calculate_cross_entropy_loss(self.y_train, train_proba),
            'val_loss': calculate_cross_entropy_loss(self.y_val, val_proba),
            'test_loss': calculate_cross_entropy_loss(self.y_test, test_proba),
            'train_acc': accuracy_score(self.y_train, train_preds),
            'val_acc': accuracy_score(self.y_val, val_preds),
            'test_acc': accuracy_score(self.y_test, test_preds),
            'lr': model.optimizer.param_groups[0]['lr']
        }

        # --- 2. Log History ---
        self.history['epoch'].append(epoch + 1)
        for key, value in metrics.items():
            self.history[key].append(value)

        # --- 3. Print Status ---
        print(f"Epoch {epoch+1:03d}/{self.network.max_epochs} | "
              f"Time: {time.time() - self.start_time:.1f}s | "
              f"LR: {metrics['lr']:.1e}")
        print(f"  ├─ Train | Loss: {metrics['train_loss']:.4f} | F1: {metrics['train_f1']:.4f}")
        print(f"  ├─ Valid | Loss: {metrics['val_loss']:.4f} | F1: {metrics['val_f1']:.4f}")
        print(f"  └─ Test  | Loss: {metrics['test_loss']:.4f} | F1: {metrics['test_f1']:.4f}  (for observation)")

        # --- 4. Early Stopping Logic ---
        val_f1 = metrics['val_f1']
        if val_f1 > self.best_val_f1 + self.min_delta:
            self.best_val_f1 = val_f1
            self.wait = 0
            best_model_path = os.path.join(self.checkpoint_dir, "best_model.zip")
            model.save_model(best_model_path)
            print(f"    └─ ✅ New best validation F1: {self.best_val_f1:.4f}. Best model saved.")
        else:
            self.wait += 1
            print(f"    └─ ⏳ Validation F1 did not improve for {self.wait}/{self.patience} epochs.")
            if self.wait >= self.patience:
                self.stopped_epoch = epoch
                self.stop_training = True # Signal TabNet to stop
                print(f"\n🛑 Early Stopping triggered at epoch {epoch + 1}. Best F1: {self.best_val_f1:.4f}.")

        # --- 5. Periodic Checkpoint Saving ---
        if (epoch + 1) % self.checkpoint_frequency == 0 and self.wait > 0: # Avoid saving if best model was just saved
            checkpoint_path = os.path.join(self.checkpoint_dir, f"epoch_{epoch+1:04d}.zip")
            model.save_model(checkpoint_path)
            print(f"💾 Checkpoint saved for epoch {epoch+1}.")
        print("-" * 80)

    def get_history(self):
        """Returns the collected training history."""
        return self.history


class TabNetTrainer:
    """A streamlined trainer for a single TabNet configuration."""
    def __init__(self, config_name, data):
        self.config_name = config_name
        self.config = TABNET_COMPLETE_CONFIGS[config_name]
        self.data = data
        self.model = None
        self.callback = None
        self.training_time = 0

    def build_model(self):
        """Builds the TabNetClassifier model from the configuration."""
        print(f"  - Building model '{self.config_name}'...")
        self.model = TabNetClassifier(
            optimizer_fn=torch.optim.Adam,
            optimizer_params=dict(lr=self.config['learning_rate'], weight_decay=self.config['weight_decay']),
            scheduler_fn=torch.optim.lr_scheduler.ReduceLROnPlateau,
            scheduler_params=dict(mode='max', factor=self.config['scheduler_factor'], patience=self.config['scheduler_patience'], min_lr=1e-6),
            device_name=DEVICE,
            **{k: v for k, v in self.config.items() if k in ['n_d', 'n_a', 'n_steps', 'gamma', 'n_independent', 'n_shared', 'lambda_sparse', 'mask_type', 'momentum', 'clip_value']}
        )

    def train(self, checkpoint_frequency=5):
        """Runs the training process."""
        if self.model is None:
            self.build_model()

        print(f"\n🚀 Starting training for {self.config_name}...")
        
        self.callback = SupervisingCallback(
            X_train=self.data['X_train'], y_train=self.data['y_train'],
            X_val=self.data['X_val'], y_val=self.data['y_val'],
            X_test=self.data['X_test'], y_test=self.data['y_test'],
            config={'config_name': self.config_name, **self.config},
            export_path=EXPORT_PATH,
            checkpoint_frequency=checkpoint_frequency
        )

        start_time = time.time()
        
        # A custom metric for TabNet's internal progress bar
        class MacroF1(Metric):
            def __init__(self): self._name, self._maximize = "macro_f1", True
            def __call__(self, y_true, y_score):
                return f1_score(y_true, np.argmax(y_score, axis=1), average='macro')

        self.model.fit(
            X_train=self.data['X_train'], y_train=self.data['y_train'],
            eval_set=[(self.data['X_val'], self.data['y_val'])],
            eval_name=['val'],
            eval_metric=['accuracy', 'logloss', MacroF1],
            max_epochs=self.config['max_epochs'],
            patience=0,  # Disable internal early stopping; our callback handles it.
            batch_size=self.config['batch_size'],
            virtual_batch_size=max(32, int(self.config['batch_size'] * self.config['virtual_batch_size_ratio'])),
            num_workers=0,
            drop_last=False,
            callbacks=[self.callback]
        )
        
        self.training_time = time.time() - start_time
        print(f"✅ Training for {self.config_name} finished in {self.training_time:.2f} seconds.")


class ExperimentManager:
    """Manages the overall experiment, running all configurations and generating reports."""
    def __init__(self, data):
        self.data = data
        self.results = []

    def run_experiment(self, configs_to_run, checkpoint_freq=5):
        """Runs the full experiment for a list of configurations."""
        print("\n" + "="*80)
        print("🔬 STARTING FULL EXPERIMENT")
        print("="*80)
        
        for config_name in configs_to_run:
            trainer = TabNetTrainer(config_name, self.data)
            trainer.train(checkpoint_frequency=checkpoint_freq)
            
            # Save the final results
            history = trainer.callback.get_history()
            if history['epoch']: # Check if training actually ran
                final_epoch_idx = np.argmax(history['val_f1'])
                self.results.append({
                    'config_name': config_name,
                    'config': trainer.config,
                    'best_epoch': history['epoch'][final_epoch_idx],
                    'best_val_f1': history['val_f1'][final_epoch_idx],
                    'corresponding_test_f1': history['test_f1'][final_epoch_idx],
                    'training_time_seconds': trainer.training_time,
                    'history': history
                })

        self.generate_reports()

    def generate_reports(self):
        """Generates and saves summary reports and visualizations."""
        if not self.results:
            print("❌ No results to report.")
            return

        print("\n" + "="*80)
        print("📊 GENERATING FINAL REPORTS")
        print("="*80)

        # 1. Create a Pandas DataFrame for easy comparison
        report_df = pd.DataFrame([{
            'Config': r['config_name'],
            'Validation F1': r['best_val_f1'],
            'Test F1': r['corresponding_test_f1'],
            'Best Epoch': r['best_epoch'],
            'Training Time (min)': r['training_time_seconds'] / 60
        } for r in self.results])
        
        report_df = report_df.sort_values(by='Validation F1', ascending=False).reset_index(drop=True)
        
        print("\n--- Performance Summary ---")
        print(report_df.to_string())
        
        # Save to CSV
        csv_path = os.path.join(EXPORT_PATH, 'experiment_summary.csv')
        report_df.to_csv(csv_path, index=False)
        print(f"\n💾 Summary saved to {csv_path}")

        # 2. Generate visualization plots
        self.plot_summary_visuals(report_df)

    def plot_summary_visuals(self, report_df):
        """Creates and saves summary plots."""
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('TabNet Experiment Summary', fontsize=16, fontweight='bold')

        # F1 Score Comparison
        sns.barplot(x='Config', y='Validation F1', data=report_df, ax=axes[0, 0], palette='viridis')
        axes[0, 0].set_title('Validation F1 Score Comparison')
        axes[0, 0].tick_params(axis='x', rotation=45)

        # Training Time Comparison
        sns.barplot(x='Config', y='Training Time (min)', data=report_df, ax=axes[0, 1], palette='plasma')
        axes[0, 1].set_title('Training Time Comparison')
        axes[0, 1].tick_params(axis='x', rotation=45)

        # Training Curves for Top 2 Models
        for i, config_name in enumerate(report_df['Config'].head(2)):
            result = next(r for r in self.results if r['config_name'] == config_name)
            history = result['history']
            ax = axes[1, i]
            ax.plot(history['epoch'], history['train_f1'], label='Train F1', color='blue')
            ax.plot(history['epoch'], history['val_f1'], label='Validation F1', color='green')
            ax.set_title(f'F1 Curves: {config_name}')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Macro F1 Score')
            ax.legend()
            ax.grid(True, linestyle='--')

        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plot_path = os.path.join(EXPORT_PATH, 'visualizations', 'experiment_summary.png')
        plt.savefig(plot_path, dpi=300)
        plt.show()
        print(f"💾 Visualization saved to {plot_path}")


# --- Main Execution Block ---

if __name__ == "__main__":
    # Define the path to your data file
    data_file_path = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/DATA/TRAIN38_no_label43.mat'
    
    # Check if the data file exists
    if not os.path.exists(data_file_path):
        print(f"❌ ERROR: Data file not found at '{data_file_path}'")
        print("Please update the 'data_file_path' variable with the correct location of your data.")
    else:
        # Load data
        X_train, y_train, X_val, y_val, X_test, y_test = load_and_preprocess_data(data_file_path)
        
        # Package data for easy passing
        data_dict = {
            'X_train': X_train, 'y_train': y_train,
            'X_val': X_val, 'y_val': y_val,
            'X_test': X_test, 'y_test': y_test,
        }

        # --- User Interaction ---
        print("\n" + "="*80)
        print("🎮 TabNet Experiment Runner")
        print("="*80)
        print("Select an experiment to run:")
        print("  1. Quick Test (Runs 'micro' and 'small' configs with fewer epochs)")
        print("  2. Full Experiment (Runs all 5 configs)")
        print("="*80)
        
        choice = input("Enter your choice (1/2): ").strip()

        if choice == '1':
            print("\n🚀 Launching Quick Test...")
            # Modify configs for a quick run
            original_configs = deepcopy(TABNET_COMPLETE_CONFIGS)
            quick_configs_to_run = ['micro_config', 'small_config']
            for name in quick_configs_to_run:
                TABNET_COMPLETE_CONFIGS[name]['max_epochs'] = 15
                TABNET_COMPLETE_CONFIGS[name]['patience'] = 5
            
            manager = ExperimentManager(data_dict)
            manager.run_experiment(configs_to_run=quick_configs_to_run, checkpoint_freq=5)
            
            # Restore original configs
            TABNET_COMPLETE_CONFIGS = original_configs

        elif choice == '2':
            print("\n🚀 Launching Full Experiment...")
            configs_to_run = ['xlarge_config', 'large_config', 'medium_config', 'small_config', 'micro_config']
            manager = ExperimentManager(data_dict)
            manager.run_experiment(configs_to_run=configs_to_run, checkpoint_freq=5)

        else:
            print("❌ Invalid choice. Exiting.")

        print("\n🎉 Experiment finished.")
